# CommaSeparatedListOutputParser - 쉼표 구분 리스트로 변환해요

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- `CommaSeparatedListOutputParser`로 LLM 출력을 Python `list`로 변환할 수 있어요
- 스트리밍 모드에서 항목이 완성될 때마다 실시간으로 출력할 수 있어요
- 키워드 추출, 추천 시스템, 브레인스토밍 등 다양한 실무 시나리오에 적용할 수 있어요

## 사전 지식

- **CommaSeparatedList(쉼표 구분 목록)**: `"항목1, 항목2, 항목3"` 형태의 텍스트를 `["항목1", "항목2", "항목3"]`으로 변환하는 가장 단순한 리스트 파싱 방식이에요

## 파이프라인 흐름

```mermaid
flowchart LR
    A["PromptTemplate<br/>(형식 지침 포함)"] -->|"프롬프트 생성"| B["ChatOpenAI<br/>(LLM)"]
    B -->|"'항목1, 항목2, ...' 텍스트 반환"| C["CommaSeparatedList<br/>OutputParser"]
    C -->|"split + strip 처리"| D["Python list<br/>항목1, 항목2, ..."]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class B,C process
    class D output
```

## 언제 사용하면 좋을까요?

- 키워드, 해시태그, 태그를 **추출**할 때
- 추천 아이템 목록을 **생성**할 때
- 태스크나 주제를 **나열**할 때
- Pydantic 스키마 없이 **빠르게 배열 데이터**를 얻고 싶을 때

> **제한 사항**: 중첩 구조(리스트 안의 딕셔너리 등)는 지원하지 않아요. 복잡한 구조가 필요하면 노트북 02의 `JsonOutputParser`를 사용하세요.

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

## 기본 사용법

가장 간단한 형태의 `CommaSeparatedListOutputParser` 사용 예제입니다.

> 🎯 **강의 포인트**: `CommaSeparatedListOutputParser`는 Pydantic 모델 없이 즉시 사용할 수 있는 가장 간단한 리스트 파서예요. `get_format_instructions()`가 반환하는 지시문이 매우 짧다는 것을 확인해보세요.

In [2]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1단계: CommaSeparatedListOutputParser 초기화
output_parser = CommaSeparatedListOutputParser()

# 2단계: 형식 지침 확인
format_instructions = output_parser.get_format_instructions()
print("=" * 60)
print("📋 파서가 생성한 형식 지침")
print("=" * 60)
print(format_instructions)
print()


📋 파서가 생성한 형식 지침
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`



In [3]:
# ---------------------------------------------------
# 프롬프트 템플릿 설정 및 체인 구성, 실행
# ---------------------------------------------------

# 1단계: 프롬프트 템플릿 설정
# partial_variables: 템플릿 생성 시 일부 변수를 미리 고정
# 🎯 강의 포인트: partial_variables 와 partial()은 같은 역할 — 취향에 따라 선택
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

# 2단계: 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3단계: 체인 구성
# CommaSeparatedListOutputParser: "항목1, 항목2, 항목3" → ["항목1", "항목2", "항목3"]
chain = prompt | model | output_parser

# 4단계: 실행
result = chain.invoke({"subject": "한국의 유명한 산"})

print("=" * 60)
print("🏔️ 결과 (Python 리스트)")
print("=" * 60)
print(f"타입: {type(result)}")
print(f"항목 수: {len(result)}")
print(f"\n목록:")
for i, item in enumerate(result, 1):
    print(f"  {i}. {item}")

🏔️ 결과 (Python 리스트)
타입: <class 'list'>
항목 수: 5

목록:
  1. 한라산
  2. 설악산
  3. 지리산
  4. 태백산
  5. 덕유산


---

## 스트리밍: 각 항목이 완성될 때마다 출력해요

`CommaSeparatedListOutputParser`는 스트리밍을 지원해요. `stream()` 사용 시 쉼표로 항목이 구분될 때마다 새 항목을 즉시 반환해요.

> **실무 팁**: 생성 목록이 길 때 스트리밍을 쓰면 사용자가 기다리는 동안 이미 완성된 항목부터 화면에 표시할 수 있어요.

In [4]:
# ---------------------------------------------------
# 스트리밍 출력 - 항목이 완성될 때마다 즉시 반환
# ---------------------------------------------------

print("=" * 60)
print("🔄 스트리밍 출력")
print("=" * 60)

# stream(): CommaSeparatedListOutputParser는 쉼표 구분 시마다 새 항목을 즉시 반환
# ⚠️ 주의: 각 chunk는 완성된 항목이 추가된 전체 리스트를 담고 있음
# (마지막 chunk만 최종 완성 목록)
for chunk in chain.stream({"subject": "한국의 전통 음식"}):
    print(f"항목: {chunk}")

🔄 스트리밍 출력
항목: ['김치']
항목: ['비빔밥']
항목: ['불고기']
항목: ['잡채']
항목: ['떡볶이']


## 실용 예제 1: 블로그 포스트 키워드 추출

블로그 포스트에서 SEO 키워드를 추출하는 시나리오입니다.

> 💡 **실무 팁**: 키워드 추출처럼 단순 문자열 목록이 필요한 태스크에는 `CommaSeparatedListOutputParser`가 가장 가볍고 빠른 선택이에요. Pydantic 스키마 없이 즉시 사용할 수 있어요.

In [5]:
# ---------------------------------------------------
# 실용 예제 1: 블로그 포스트 SEO 키워드 추출
# ---------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

# 키워드 추출용 프롬프트
keyword_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 SEO 전문가입니다. 주어진 텍스트에서 핵심 키워드를 추출하세요."),
    ("user", """다음 텍스트에서 SEO에 효과적인 키워드 5-7개를 추출하세요.

텍스트:
{text}

{format_instructions}""")
])

# partial()로 형식 지침 주입
keyword_prompt = keyword_prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

# 체인 구성
keyword_chain = keyword_prompt | model | output_parser

# 실행
blog_text = """
인공지능 기술이 빠르게 발전하면서 챗봇 개발이 더욱 쉬워졌습니다.
특히 LangChain 프레임워크는 대화형 AI 애플리케이션 구축을 간소화합니다.
Python을 사용하여 자연어 처리 파이프라인을 만들 수 있으며,
GPT 모델과의 통합도 매우 간단합니다.
"""

keywords = keyword_chain.invoke({"text": blog_text})

print("\n" + "=" * 60)
print("🔍 추출된 SEO 키워드")
print("=" * 60)
# 반환된 list를 순회하며 출력
for i, keyword in enumerate(keywords, 1):
    print(f"#{i} {keyword}")


🔍 추출된 SEO 키워드
#1 인공지능
#2 챗봇 개발
#3 LangChain
#4 대화형 AI
#5 자연어 처리
#6 Python
#7 GPT 모델


## 실용 예제 2: 영화 추천 시스템

사용자의 선호도를 기반으로 영화를 추천하는 시나리오입니다.


In [6]:
# 영화 추천 프롬프트
movie_prompt = PromptTemplate.from_template(
    """당신은 영화 추천 전문가입니다.

사용자 정보:
- 좋아하는 장르: {genre}
- 선호하는 분위기: {mood}
- 최근 본 영화: {recent_movie}

위 정보를 바탕으로 추천할 영화 제목 5개를 나열하세요.

{format_instructions}"""
)

movie_prompt = movie_prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

# 체인 구성
movie_chain = movie_prompt | model | output_parser

# 실행
recommendations = movie_chain.invoke({
    "genre": "SF, 스릴러",
    "mood": "긴장감 넘치는",
    "recent_movie": "인터스텔라"
})

print("=" * 60)
print("🎬 추천 영화 목록")
print("=" * 60)
for i, movie in enumerate(recommendations, 1):
    print(f"{i}. {movie}")


🎬 추천 영화 목록
1. 덩케르크
2. 블레이드 러너 2049
3. 에이리언: 커버넌트
4. 소스 코드
5. 테넷


## 실용 예제 3: 프로젝트 태스크 분해

큰 프로젝트를 작은 태스크로 분해하는 시나리오입니다.


In [7]:
# 태스크 분해 프롬프트
task_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 프로젝트 관리 전문가입니다."),
    ("user", """다음 프로젝트를 5-8개의 구체적인 태스크로 분해하세요.

프로젝트: {project}

각 태스크는 간결하게 작성하세요.

{format_instructions}""")
])

task_prompt = task_prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

# 체인 구성
task_chain = task_prompt | model | output_parser

# 실행
tasks = task_chain.invoke({
    "project": "온라인 쇼핑몰 웹사이트 개발"
})

print("\n" + "=" * 60)
print("📋 프로젝트 태스크 목록")
print("=" * 60)
print("프로젝트: 온라인 쇼핑몰 웹사이트 개발\n")
for i, task in enumerate(tasks, 1):
    print(f"✓ Task {i}: {task}")



📋 프로젝트 태스크 목록
프로젝트: 온라인 쇼핑몰 웹사이트 개발

✓ Task 1: 요구사항 분석
✓ Task 2: 디자인 프로토타입 제작
✓ Task 3: 프론트엔드 개발
✓ Task 4: 백엔드 개발
✓ Task 5: 데이터베이스 설계 및 구축
✓ Task 6: 결제 시스템 통합
✓ Task 7: 테스트 및 버그 수정
✓ Task 8: 배포 및 유지보수 계획 수립


## 실용 예제 4: 학습 주제 생성

특정 분야의 학습 로드맵을 생성하는 시나리오입니다.

> 🔑 **핵심 개념**: `CommaSeparatedListOutputParser`는 어떤 프롬프트와도 결합할 수 있어요. 중요한 것은 `get_format_instructions()`를 프롬프트에 반드시 포함시키는 것이에요. 이 지시문이 없으면 LLM이 쉼표 구분 형식으로 답하지 않을 수 있어요.

In [8]:
# 학습 주제 생성 프롬프트
learning_prompt = PromptTemplate.from_template(
    """{level} 수준에서 {subject}를 배우기 위한 핵심 학습 주제 6개를 순서대로 나열하세요.

각 주제는 간결하게 작성하세요.

{format_instructions}"""
)

learning_prompt = learning_prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

# 체인 구성
learning_chain = learning_prompt | model | output_parser

# 실행
topics = learning_chain.invoke({
    "level": "초급",
    "subject": "데이터 과학"
})

print("=" * 60)
print("📚 학습 로드맵")
print("=" * 60)
print("분야: 데이터 과학 (초급)\n")
for i, topic in enumerate(topics, 1):
    print(f"Step {i}: {topic}")


📚 학습 로드맵
분야: 데이터 과학 (초급)

Step 1: 데이터 수집
Step 2: 데이터 전처리
Step 3: 데이터 탐색적 분석
Step 4: 데이터 시각화
Step 5: 기계 학습 기초
Step 6: 모델 평가 및 개선


## 다양한 활용 예제

`CommaSeparatedListOutputParser`는 다양한 시나리오에서 활용할 수 있습니다.


In [9]:
# 간단한 프롬프트로 다양한 목록 생성
simple_prompt = PromptTemplate.from_template(
    "{request}\n\n{format_instructions}"
)

simple_prompt = simple_prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

simple_chain = simple_prompt | model | output_parser

# 1. 브레인스토밍
print("\n" + "=" * 60)
print("💡 브레인스토밍: 마케팅 아이디어")
print("=" * 60)
ideas = simple_chain.invoke({"request": "신제품 출시를 위한 마케팅 아이디어 5가지를 제안하세요."})
for i, idea in enumerate(ideas, 1):
    print(f"{i}. {idea}")

# 2. 체크리스트 생성
print("\n" + "=" * 60)
print("✅ 체크리스트: 여행 준비물")
print("=" * 60)
checklist = simple_chain.invoke({"request": "3박 4일 해외여행에 필요한 필수 준비물 7가지를 나열하세요."})
for item in checklist:
    print(f"☐ {item}")

# 3. 카테고리 분류
print("\n" + "=" * 60)
print("🏷️ 상품 카테고리")
print("=" * 60)
categories = simple_chain.invoke({"request": "온라인 쇼핑몰의 주요 상품 카테고리 6개를 나열하세요."})
for i, category in enumerate(categories, 1):
    print(f"[{i}] {category}")



💡 브레인스토밍: 마케팅 아이디어
1. 소셜 미디어 챌린지
2. 인플루언서 협업
3. 한정판 프로모션
4. 체험 이벤트
5. 사용자 생성 콘텐츠 캠페인

✅ 체크리스트: 여행 준비물
☐ 여권
☐ 항공권
☐ 여행자 보험
☐ 현금 및 신용카드
☐ 의류 및 개인 용품
☐ 전자기기 및 충전기
☐ 약품 및 응급처치 키트

🏷️ 상품 카테고리
[1] 의류
[2] 전자제품
[3] 가정용품
[4] 뷰티/화장품
[5] 스포츠/레저
[6] 식품/음료


## 핵심 요약

| 항목 | 설명 |
|------|------|
| **역할** | 쉼표 구분 텍스트 → Python `list` 변환 |
| **설정** | `CommaSeparatedListOutputParser()` 한 줄, 스키마 불필요 |
| **스트리밍** | 항목 완성 시마다 즉시 반환 |
| **제한** | 단순 문자열 리스트만 지원, 중첩 구조 불가 |

> ⚠️ **자주 하는 실수**: 항목 안에 쉼표가 포함된 경우 (예: "서울, 경기") 파서가 두 항목으로 잘못 분리할 수 있어요. 이럴 때는 `JsonOutputParser`로 전환하세요.

### 활용 시나리오별 비교

| 시나리오 | 추천 파서 | 이유 |
|----------|-----------|------|
| 키워드·해시태그 추출 | CommaSeparatedList | 단순 문자열 목록이면 충분 |
| 영화·책 추천 목록 | CommaSeparatedList | 제목만 필요할 때 |
| 추천 + 상세 정보 | JsonOutputParser | 제목·평점·설명 등 구조 필요 |
| 우선순위 분류 포함 | PydanticOutputParser | 타입 검증 + Enum 조합 |

## 다음 노트북 예고

**노트북 04 - 특수 타입 파싱** 에서는 `datetime`(날짜/시간)과 `Enum`(열거형) 타입을 Pydantic과 결합해서 파싱하는 방법을 배워요. 날짜 추출, 카테고리 분류, 감정 분석 등에 유용해요.

## 연습 문제

다음 연습 문제를 통해 `CommaSeparatedListOutputParser`의 활용법을 직접 실습해봅시다.

### 연습 1: 해시태그 생성기

`CommaSeparatedListOutputParser`를 사용하여 블로그 포스트 제목에서 **SNS 해시태그**를 자동 생성하는 체인을 만들어보세요.

**요구사항:**
- `ChatPromptTemplate`으로 시스템 메시지("당신은 SNS 마케팅 전문가입니다.")를 포함하는 프롬프트를 구성하세요
- 입력 변수: `post_title` (블로그 포스트 제목)
- 해시태그 7개를 생성하도록 프롬프트에 명시하세요
- 결과 리스트의 각 항목 앞에 `#`을 붙여 출력하세요
- 테스트 제목: "초보자를 위한 LangChain 입문 가이드: AI 앱 만들기"

In [10]:
# ============================================================
# TODO: 해시태그 생성기 만들기
# 힌트: ChatPromptTemplate으로 시스템 메시지를 추가하고,
#       output_parser.get_format_instructions()를 partial()로 주입하세요
# 예상 결과: list 타입으로 해시태그 7개가 반환되고 앞에 # 붙여 출력
# ============================================================

# 연습 1 풀이

from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 1단계: 파서 및 모델 초기화
# CommaSeparatedListOutputParser: 별도 스키마 없이 즉시 사용 가능
output_parser = CommaSeparatedListOutputParser()
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2단계: 해시태그 생성 프롬프트
hashtag_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 SNS 마케팅 전문가입니다."),
    ("user", """다음 블로그 포스트 제목에 어울리는 SNS 해시태그 7개를 생성하세요.
해시태그는 '#' 기호 없이 키워드만 작성하세요.

포스트 제목: {post_title}

{format_instructions}""")
])

hashtag_prompt = hashtag_prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

# 3단계: 체인 구성 및 실행
hashtag_chain = hashtag_prompt | model | output_parser

post_title = "초보자를 위한 LangChain 입문 가이드: AI 앱 만들기"
hashtags = hashtag_chain.invoke({"post_title": post_title})

print("=" * 60)
print(f"포스트: {post_title}")
print("=" * 60)
print(f"생성된 해시태그 ({len(hashtags)}개):")
# 각 항목 앞에 # 붙여 SNS 해시태그 형식으로 출력
for tag in hashtags:
    print(f"  #{tag}")

포스트: 초보자를 위한 LangChain 입문 가이드: AI 앱 만들기
생성된 해시태그 (7개):
  #LangChain
  #AI앱
  #초보자
  #인공지능
  #프로그래밍
  #개발자
  #기술가이드


### 연습 2: 스트리밍 퀴즈 생성기

`CommaSeparatedListOutputParser`와 `stream()` 메서드를 사용하여 특정 주제에 대한 **퀴즈 문제**를 스트리밍으로 생성하는 체인을 만들어보세요.

**요구사항:**
- `PromptTemplate`을 사용하세요
- 입력 변수: `subject` (과목), `count` (문제 수)
- `stream()` 메서드로 각 문제가 생성될 때마다 실시간 출력하세요
- 테스트: 과목 "한국사", 문제 수 5

In [11]:
# ============================================================
# TODO: 스트리밍 퀴즈 생성기 만들기
# 힌트: PromptTemplate에 subject, count 변수를 정의하고
#       chain.stream()으로 실행하세요. 각 chunk는 리스트입니다
# 예상 결과: 각 퀴즈 문제가 완성될 때마다 실시간으로 출력됨
# ============================================================

# 연습 2 풀이

from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 1단계: 파서 및 모델 초기화
output_parser = CommaSeparatedListOutputParser()
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2단계: 퀴즈 생성 프롬프트
quiz_prompt = PromptTemplate.from_template(
    """{subject} 과목에서 {count}개의 간단한 퀴즈 문제를 만들어주세요.
각 문제는 한 문장으로 간결하게 작성하세요.

{format_instructions}"""
)

quiz_prompt = quiz_prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

# 3단계: 체인 구성
quiz_chain = quiz_prompt | model | output_parser

# 4단계: 스트리밍 실행 - 문제가 완성될 때마다 즉시 출력
print("=" * 60)
print("한국사 퀴즈 (스트리밍)")
print("=" * 60)

question_num = 1
for chunk in quiz_chain.stream({"subject": "한국사", "count": 5}):
    # 각 chunk는 리스트이므로 항목별 출력
    for item in chunk:
        print(f"Q{question_num}. {item}")
        question_num += 1

한국사 퀴즈 (스트리밍)
Q1. 고조선의 건국 신화에 등장하는 인물은 누구인가?
Q2. 삼국시대의 세 나라를 나열하시오.
Q3. 고려시대의 대표적인 유교 성리학자는 누구인가?
Q4. 조선시대의 왕권을 강화한 법전의 이름은 무엇인가?
Q5. 1919년 3.1운동의 주도 세력은 어떤 단체였는가?
